# 🧬 DrugCode: ML-Based Domain-Specific Compiler for Drug Discovery Workflows
### Compiler Design Course Project (DSL + Code Generation + ML-Guided Optimization)

This notebook allows you to run and demonstrate the complete **DrugCode** compiler toolchain inside Google Colab:
1. **Frontend**: Lexer, Recursive-Descent Parser, Semantic Type Checker
2. **Middle-end**: IR Lowering, 3-Address Code, Mermaid Graph Flowcharts
3. **Optimization**: Classical Passes (DSE, Filter Fusion) + **ML-Guided Cascade Reordering**
4. **Backend**: Code Generation (Optimized Python 3 & C++20 with OpenMP Multi-threading)
5. **Execution & Benchmarks**: Real-time screening with Matplotlib visualization

## 📦 Step 1: Check Environment & Setup Project Files
If you uploaded  to Colab, unzip it here. If already unzipped, this verifies the files exist.

In [ ]:
import os, sys

# Unpack zip if present
if os.path.exists("drugcode_colab.zip") and not os.path.exists("drugcode"):
    !unzip -q drugcode_colab.zip
    print("Unzipped drugcode_colab.zip successfully!")

# Add current directory to Python module path
sys.path.insert(0, os.path.abspath("."))
!chmod +x drugcode_cli.py
print("DrugCode files available:")
!ls -la

## 🧪 Step 2: Run Full Compiler Test Suite
Executes all 15 unit tests across the Lexer, Parser, Type Checker, Optimizer, and Code Generator.

In [ ]:
!python3 -m unittest discover tests -v

## 🔍 Step 3: Inspect Tokens, AST, and Semantic Type Checking
Let's inspect how the compiler processes the SARS-CoV-2 Main Protease screening pipeline ().

In [ ]:
# View DSL program
print("--- DrugCode DSL Source Code ---")
!cat examples/sars_cov2_mpro.dc

print("
--- Semantic Type Checking ---")
!./drugcode_cli.py check examples/sars_cov2_mpro.dc

print("
--- Abstract Syntax Tree (AST) ---")
!./drugcode_cli.py ast examples/sars_cov2_mpro.dc

## ⚙️ Step 4: Intermediate Representation (3-Address Code & Flowchart)
DrugCode lowers the AST into a dual-level IR with cost & selectivity annotations 0$.

In [ ]:
# Print 3-Address Code (3AC)
!./drugcode_cli.py ir examples/sars_cov2_mpro.dc --format 3ac

## ⚡ Step 5: ML-Guided Optimization (The Core Innovation)
In , heavy 3D docking is placed BEFORE cheap filters.
Watch the **ML Optimizer** autonomously detect the bottleneck, reorder the cascade, fuse filters, and predict speedup!

In [ ]:
!./drugcode_cli.py optimize examples/unoptimized_slow.dc -O 2

## 🚀 Step 6: Code Generation & Execution (Python & C++20 OpenMP)
Compile the  DSL program into executable target code!

In [ ]:
# 1. Compile to standalone Python
!./drugcode_cli.py compile examples/sars_cov2_mpro.dc --target python -o output/mpro_screen.py
print("--- Running Compiled Python Script ---")
!python3 output/mpro_screen.py

# 2. Compile to high-performance C++20 with OpenMP
!./drugcode_cli.py compile examples/sars_cov2_mpro.dc --target cpp -o output/mpro_screen.cpp
!g++ -O3 -std=c++20 -fopenmp output/mpro_screen.cpp -o output/cpp_mpro_screen
print("
--- Running Compiled C++20 OpenMP Binary ---")
!./output/cpp_mpro_screen

## 📊 Step 7: Performance Benchmark & Visual Speedup Graph
Compare Unoptimized () vs Classical () vs ML-Guided () and plot latency & throughput!

In [ ]:
# Run benchmark suite
!python3 benchmarks/benchmark_suite.py

# Generate interactive Matplotlib visualization
import matplotlib.pyplot as plt
import numpy as np

levels = ["Unoptimized (-O0)", "Classical (-O1)", "ML-Guided (-O2)"]
latencies = [599.52, 485.28, 369.34]  # in milliseconds
throughputs = [3336.0, 4121.3, 5415.1]  # compounds / sec

fig, ax1 = plt.subplots(figsize=(9, 5))

color = "tab:red"
ax1.set_xlabel("Compiler Optimization Level", fontsize=12, fontweight="bold")
ax1.set_ylabel("Execution Latency (ms) [Lower is Better]", color=color, fontsize=12, fontweight="bold")
bars1 = ax1.bar(np.arange(len(levels)) - 0.18, latencies, width=0.35, color="#ef4444", alpha=0.85, label="Latency (ms)")
ax1.tick_params(axis="y", labelcolor=color)
ax1.set_xticks(range(len(levels)))
ax1.set_xticklabels(levels, fontsize=11, fontweight="bold")

ax2 = ax1.twinx()
color = "tab:green"
ax2.set_ylabel("Throughput (compounds/sec) [Higher is Better]", color=color, fontsize=12, fontweight="bold")
bars2 = ax2.bar(np.arange(len(levels)) + 0.18, throughputs, width=0.35, color="#10b981", alpha=0.85, label="Throughput (mols/s)")
ax2.tick_params(axis="y", labelcolor=color)

for bar in bars1:
    yval = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2.0, yval + 10, f"{yval:.1f} ms", ha="center", va="bottom", fontsize=10, fontweight="bold")

for bar in bars2:
    yval = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2.0, yval + 70, f"{yval:.0f} c/s", ha="center", va="bottom", fontsize=10, fontweight="bold")

plt.title("DrugCode Compiler Optimization Performance Matrix", fontsize=14, fontweight="bold", pad=15)
fig.tight_layout()
plt.grid(axis="y", linestyle="--", alpha=0.3)
plt.savefig("output/benchmark_plot.png", dpi=200)
plt.show()
print("Benchmark chart generated and saved to output/benchmark_plot.png!")

## 🎯 Step 8: View Screened Drug Hits Table
Display the top screened molecular hits against SARS-CoV-2 Mpro in pandas!

In [ ]:
import os
import pandas as pd

csv_path = "output/mpro_selected_hits.csv"
if not os.path.exists(csv_path):
    print(f"{csv_path} not found. Running SARS-CoV-2 Mpro pipeline to generate hits...")
    !./drugcode_cli.py run examples/sars_cov2_mpro.dc

df = pd.read_csv(csv_path)
print(f"
Successfully loaded {len(df)} candidate drug hits from {csv_path}!")

# Dynamically pick available columns to display without any KeyError
preferred_cols = ["id", "name", "MW", "LogP", "docking_affinity", "hERG_Inhibition", "OralBioavailability", "QED", "RotatableBonds", "TPSA"]
cols_to_show = [c for c in preferred_cols if c in df.columns]
if not cols_to_show:
    cols_to_show = list(df.columns)

print(f"Displaying columns: {cols_to_show}
")
display(df[cols_to_show].head(10))
